# **User Defined Function (UDf)**

In [0]:
data = [ ("Finance",10000),("Marketing",20000),("Sales",30000),("IT",40000)]
df = spark.createDataFrame(data, ["department", "salary"])
display(df)

In [0]:
from pyspark.sql.types import *

def addone(a):
    return a + 100

def multiply(a):
    return a * 2

def add(a,b):
    return a + b

addone_df = udf(addone, IntegerType())
multiply_df = udf(multiply, IntegerType())
add_df = udf(add, IntegerType())

In [0]:
df.select("department","salary",addone_df("salary"),multiply_df("salary"),add_df(addone_df("salary"),multiply_df("salary"))).show()

# **DataFrame.transform()**

In [0]:
data1 = [(1, "ali", 10000), (2, "veli", 20000), (3, "kenan", 30000), (4, "murat", 40000)]
df1 = spark.createDataFrame(data1, ["id", "name", "salary"])
display(df1)

In [0]:
from pyspark.sql.functions import *

def uppercase(df1):
    return df1.withColumn("name", upper(df1.name))

In [0]:
df1.transform(uppercase).show()

# **Temp View**

In [0]:
df2 = spark.read.csv("/Volumes/workspace/default/apacha-data/MegaMart.csv", header=True, inferSchema=True)
display(df2)

In [0]:
df2.createOrReplaceTempView("mart")

In [0]:
%sql
Select * from mart

In [0]:
%sql
select upper(product_name), max(price_per_unit) from mart GROUP BY upper(product_name)

# **Window Function**

In [0]:
data2 = [(1, "Amaan", "Pakistan", 50000), (2, "Bilal", "India", 60000), (3, "Charlie", "Pakistan", 90000), (4, "David", "India", 80000), (5, "Eve", "Pakistan", 90000), (6, "Fiona", "India", 100000), (7, "George", "Pakistan", 110000), (8, "Hannah", "India", 120000), (9, "Ivan", "Pakistan", 130000), (10, "Jasmine", "India", 120000)]
df3 = spark.createDataFrame(data2, ["id", "name", "country", "salary"])

display(df3)

In [0]:

from pyspark.sql.window import Window
from pyspark.sql.functions import rank, col, dense_rank, row_number

window = Window.partitionBy("country").orderBy(col("salary").desc())
df_wf = (   df3.withColumn("rank", rank().over(window))
         
    .withColumn("dense_rank", dense_rank().over(window))

    .withColumn("row_number", row_number().over(window)))

display(df_wf)